### Störung – Lösung Schritt für Schritt

*41. Bundeswettbewerb Informatik (2022), 1. Runde, Aufgabe 1*

Alice und Bob tauschen Sätze aus ihrem Lieblingsbuch aus. Trudi reißt Lücken in die Sätze, ganze Wörter verschwinden. Bob
empfing zum Beispiel diesen Lückensatz:

    das … mir … … … vor

Er fand im Buch die Stelle `das kommt mir gar nicht richtig vor`, ist aber unsicher, ob es noch andere passende Stellen gibt.

#### Aufgabe 1

Schreibe ein Programm, das für einen Lückensatz nach passenden Stellen im Lieblingsbuch sucht. Wende es auf die Beispiele von
den BWINF-Webseiten an und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/`)

- `Alice_im_Wunderland.txt`: das Lieblingsbuch (3685 Zeilen)
- `stoerung0.txt` bis `stoerung5.txt`: je ein Lückensatz, Lücken als `_` (`stoerung0.txt`: `das _ mir _ _ _ vor`)

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Muster an jeder Stelle anlegen (naive Textsuche):** Man legt das Muster an Position 0, 1, 2, ... der Wortliste an und
  vergleicht Element für Element. Beim ersten Unterschied bricht man ab und probiert die nächste Position.
- **Tokenisieren:** Bevor man vergleicht, zerlegt man den Text in sinnvolle Einheiten (hier: Wörter und Satzzeichen). Das
  macht den Vergleich einfach, und die Frage „Was ist ein Wort?“ lässt sich an einer Stelle klären.
- **Modellierung:** Wir müssen festlegen, was als Treffer zählt (Groß-/Kleinschreibung, Satzzeichen, Bindestriche).

**Nützliche Funktionen und Bibliotheken**
- `re.finditer`: Wörter per regulärem Ausdruck herausschneiden (mit Position im Text)
- `str.lower()`: Groß-/Kleinschreibung ignorieren
- `' '.join(text.split())`: Zeilenumbrüche und mehrfache Leerzeichen zu einem Leerzeichen machen

## Die Idee

1. **Festlegen, was zählt.** Wir folgen den Lösungshinweisen:
   - Ein `_` steht für **genau ein ganzes Wort**.
   - Groß- und Kleinschreibung ist egal (im Buch steht `Das kommt mir ...` mit großem D).
   - **Satzzeichen** sind keine Wörter. Ein Platzhalter kann sie nie „verschlucken“; sie müssen im Muster genauso stehen wie im
     Text. Das Muster `ich muß _ clara _` passt also nicht auf `ich muß, Clara ...`.
   - Ein **Bindestrich** gehört zum Wort (`Fuß-Boden`). Steht er am Zeilenende, ist es ein Trennstrich, und die beiden Hälften
     werden zu einem Wort zusammengesetzt.
2. **Text und Muster zerlegen** in Tokens: Wörter und einzelne Satzzeichen.
3. **Muster an jeder Position anlegen** und Token für Token vergleichen.
4. **Treffer ausgeben**, und zwar so, dass man sieht, welche Wörter die Platzhalter ersetzen.

## Schritt 1: Den Text einlesen

Das Buch ist in UTF-8 gespeichert (am Dateianfang steht ein unsichtbares Zeichen, das `encoding='utf-8-sig'` entfernt).
Trennstriche am Zeilenende (Buchstabe, `-`, Zeilenende) fügen wir sofort mit einem regulären Ausdruck zusammen. In unserem Buch
kommt das gar nicht vor (die Gedankenstriche `--` am Zeilenende haben ein Leerzeichen davor), aber so wäre das Programm auch für
andere Bücher richtig. Ein Gedankenstrich bleibt dabei unberührt, weil wir nur ein `-` **direkt hinter einem Buchstaben**
zusammenfügen.

In [ ]:
import re


def lies_text(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        text = f.read()
    # Trennstrich am Zeilenende: Buchstabe, '-', Zeilenumbruch, evtl. Einrückung -> Wortteile zusammenfügen
    text = re.sub('(?<=[A-Za-zäöüÄÖÜß])-\\n\\s*', '', text)
    return text


buch = lies_text('beispieldaten/Alice_im_Wunderland.txt')
print(len(buch), 'Zeichen')
print(buch[600:760])

159416 Zeichen
eben ihn, sie drängten ihn,
    Ein Mährchen zu erzählen.

    Die erste gab's Commandowort;
    O schnell, o fange an!
    Und mach' es so, die Zweite bat,
   


## Schritt 2: In Tokens zerlegen

Ein **Wort** besteht aus Buchstaben (auch ä, ö, ü, ß), die durch einzelne Bindestriche verbunden sein dürfen. Ein **Satzzeichen**
ist jedes andere Zeichen, das kein Leerraum ist (also auch `_`, Klammern, Anführungszeichen, Ziffern). Das erledigt ein
regulärer Ausdruck mit zwei Alternativen, die durch `|` getrennt sind:

- `[A-Za-zäöüÄÖÜß]+(?:-[A-Za-zäöüÄÖÜß]+)*`: Buchstaben, danach beliebig oft Bindestrich plus Buchstaben
- `\S`: irgendein einzelnes Zeichen, das kein Leerraum ist

Weil `re.finditer` auch die **Position** im Text liefert, merken wir sie uns: So können wir später die Fundstelle im Original
ausgeben. Jedes Token ist ein Tupel `(kleingeschrieben, anfang, ende)`. Das Kleinschreiben erledigt die Groß-/Kleinschreibung
ein für alle Mal.

Der Platzhalter `_` im Muster erscheint dabei als Token `'_'`. Im Buch kommt `_` auch vor (als Zeichen für Kursivschrift,
z. B. `_sehr_`), aber dort ist es ein ganz normales Satzzeichen-Token. Ob ein Token ein Wort ist, erkennen wir daran, dass sein
erstes Zeichen ein Buchstabe ist (`ist_wort`).

In [ ]:
WORT = '[A-Za-zäöüÄÖÜß]+(?:-[A-Za-zäöüÄÖÜß]+)*|\\S'


def tokens_von(text):
    tokens = []
    for treffer in re.finditer(WORT, text):
        tokens.append((treffer.group().lower(), treffer.start(), treffer.end()))
    return tokens


def ist_wort(token):
    return token[0].isalpha()


buch_tokens = tokens_von(buch)
print(len(buch_tokens), 'Tokens')
print(buch_tokens[40:48])
print([t[0] for t in tokens_von('Ich muß _ Clara _! Der Fuß-Boden (_sehr_)')])

33565 Tokens
[('o', 256, 257), ('schöner', 258, 265), (',', 265, 266), ('goldner', 267, 274), ('nachmittag', 275, 285), (',', 285, 286), ('wo', 291, 293), ('flut', 294, 298)]
['ich', 'muß', '_', 'clara', '_', '!', 'der', 'fuß-boden', '(', '_', 'sehr', '_', ')']


## Schritt 3: Passt ein Token zum Muster?

Die Regeln aus den Lösungshinweisen (Tabelle 1):

| Mustertoken | Texttoken | passt? |
|---|---|---|
| `_` | ein Wort | ja |
| `_` | ein Satzzeichen | nein |
| Wort / Satzzeichen | gleiches Token (klein geschrieben) | ja |
| sonst | | nein |

Weil Text und Muster schon klein geschrieben sind, reicht dafür ein Vergleich. Nur der Platzhalter braucht eine Sonderregel.

In [ ]:
def passt(muster_token, text_token):
    if muster_token[0] == '_':
        return ist_wort(text_token)       # Platzhalter: genau ein Wort, kein Satzzeichen
    return muster_token[0] == text_token[0]


m = tokens_von('das _ mir')
t = tokens_von('Das kommt, mir')
print(passt(m[0], t[0]), passt(m[1], t[1]), passt(m[1], t[2]), passt(m[2], t[3]))

True True False True


## Schritt 4: Das Muster an jeder Stelle anlegen

Für jede Startposition `i` vergleichen wir `muster[j]` mit `tokens[i + j]` für alle `j`. Sobald ein Paar nicht passt, ist
Position `i` erledigt (`break`). Passen alle, merken wir uns `i`. Die letzte mögliche Startposition ist
`len(tokens) - len(muster)`.

In [ ]:
def finde_stellen(tokens, muster):
    stellen = []
    for i in range(len(tokens) - len(muster) + 1):
        alle_passen = True
        for j in range(len(muster)):
            if not passt(muster[j], tokens[i + j]):
                alle_passen = False
                break
        if alle_passen:
            stellen.append(i)
    return stellen


muster = tokens_von('das _ mir _ _ _ vor')
stellen = finde_stellen(buch_tokens, muster)
print(stellen)

[3925]


## Schritt 5: Fundstellen lesbar ausgeben

Eine Fundstelle ist die Folge der Tokens ab `i`. Wir schneiden den Originaltext vom Anfang des ersten bis zum Ende des letzten
Tokens aus (Groß-/Kleinschreibung und Satzzeichen bleiben so erhalten) und machen aus Zeilenumbrüchen ein Leerzeichen. Dahinter
geben wir die Wörter aus, die an den Platzhaltern standen, damit man sofort sieht, was Trudi weggenommen hat.

In [ ]:
def fundstelle(text, tokens, muster, i):
    anfang = tokens[i][1]
    ende = tokens[i + len(muster) - 1][2]
    return ' '.join(text[anfang:ende].split())


def ersetzte_woerter(tokens, muster, i):
    ersetzt = []
    for j in range(len(muster)):
        if muster[j][0] == '_':
            ersetzt.append(tokens[i + j][0])
    return ersetzt


def suche(text, tokens, lueckensatz):
    muster = tokens_von(lueckensatz)
    stellen = finde_stellen(tokens, muster)
    print('Muster:', lueckensatz, '->', len(stellen), 'Treffer')
    for i in stellen:
        print('  ', fundstelle(text, tokens, muster, i), ' | Lücken:', ersetzte_woerter(tokens, muster, i))
    return stellen


suche(buch, buch_tokens, 'das _ mir _ _ _ vor')

Muster: das _ mir _ _ _ vor -> 1 Treffer
   Das kommt mir gar nicht richtig vor  | Lücken: ['kommt', 'gar', 'nicht', 'richtig']


## Schritt 6: Alle Beispiele

Wir lesen den Lückensatz aus der Datei (ein Satz in einer Zeile) und suchen.

In [ ]:
def lies_lueckensatz(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        return f.read().strip()


anzahlen = []
for nr in range(6):
    satz = lies_lueckensatz('beispieldaten/stoerung' + str(nr) + '.txt')
    anzahlen.append(len(suche(buch, buch_tokens, satz)))
    print()

Muster: das _ mir _ _ _ vor -> 1 Treffer
   Das kommt mir gar nicht richtig vor  | Lücken: ['kommt', 'gar', 'nicht', 'richtig']

Muster: ich muß _ clara _ -> 2 Treffer
   Ich muß in Clara verwandelt  | Lücken: ['in', 'verwandelt']
   Ich muß doch Clara sein  | Lücken: ['doch', 'sein']

Muster: fressen _ gern _ -> 3 Treffer
   Fressen Katzen gern Spatzen  | Lücken: ['katzen', 'spatzen']
   Fressen Katzen gern Spatzen  | Lücken: ['katzen', 'spatzen']
   Fressen Spatzen gern Katzen  | Lücken: ['spatzen', 'katzen']

Muster: das _ fing _ -> 2 Treffer
   das Spiel fing an  | Lücken: ['spiel', 'an']
   Das Publikum fing an  | Lücken: ['publikum', 'an']

Muster: ein _ _ tag -> 1 Treffer
   ein sehr schöner Tag  | Lücken: ['sehr', 'schöner']

Muster: wollen _ so _ sein -> 1 Treffer
   Wollen Sie so gut sein  | Lücken: ['sie', 'gut']



## Ergebnisse und Auswertung

- `stoerung0` (`das _ mir _ _ _ vor`): **genau eine** Stelle, `Das kommt mir gar nicht richtig vor`. Bob lag also richtig.
- `stoerung1` (`ich muß _ clara _`): **zwei** Stellen, `Ich muß in Clara verwandelt` und `Ich muß doch Clara sein`.
- `stoerung2` (`fressen _ gern _`): **drei** Treffer. `Fressen Katzen gern Spatzen` steht **zweimal** im Buch, daher erscheint es
  zweimal. Wer jede Fundstelle nur einmal ausgeben will, kann mit einer Menge (`set`) doppelte Texte entfernen.
- `stoerung3` (`das _ fing _`): `das Spiel fing an` und `Das Publikum fing an`.
- `stoerung4` (`ein _ _ tag`): `ein sehr schöner Tag`.
- `stoerung5` (`wollen _ so _ sein`): `Wollen Sie so gut sein`.

Das stimmt mit den Beispielergebnissen der Lösungshinweise überein (siehe die `assert`s im Generator). Weil Satzzeichen nicht von
Platzhaltern verschluckt werden, würde z. B. `Ich muß, Clara` nicht passen. Das ist eine Modellierungsentscheidung.

**Aufwand:** Bei `n` Tokens im Text und `m` im Muster höchstens $n \cdot m$ Vergleiche, in der Praxis kaum mehr als $n$, weil
meist schon das erste Token nicht passt. Für kurze Muster ist das für das ganze Buch (rund 33 000 Tokens) sofort erledigt.

## Für Neugierige: Mit regulären Ausdrücken

Das Buch lässt sich auch mit einem **regulären Ausdruck** durchsuchen: Wir machen aus der Tokenliste einen langen String
(Tokens durch Leerzeichen getrennt) und aus dem Muster einen Ausdruck, in dem der Platzhalter für „ein Token, das mit einem
Buchstaben beginnt“ steht. Die Position eines Treffers im String rechnen wir mit `count(' ')` in eine Tokennummer um. Das ist
eine Kontrolle: Das Ergebnis muss mit unserem Verfahren übereinstimmen.

In [ ]:
def regex_stellen(tokens, lueckensatz):
    muster = tokens_von(lueckensatz)
    teile = []
    for mt in muster:
        if mt[0] == '_':
            teile.append('[A-Za-zäöüß][^ ]*')
        else:
            teile.append(re.escape(mt[0]))
    ausdruck = '(?<![^ ])' + ' '.join(teile) + '(?![^ ])'
    langer_string = ' '.join(t[0] for t in tokens)
    stellen = []
    for treffer in re.finditer('(?=(' + ausdruck + '))', langer_string):
        stellen.append(langer_string.count(' ', 0, treffer.start()))
    return stellen


for nr in range(6):
    satz = lies_lueckensatz('beispieldaten/stoerung' + str(nr) + '.txt')
    eigene = finde_stellen(buch_tokens, tokens_von(satz))
    print(nr, eigene == regex_stellen(buch_tokens, satz))

0 True
1 True
2 True
3 True
4 True
5 True


**MiniZinc:** Für diese Aufgabe lassen wir es weg. Es gibt nichts zu optimieren und keine Bedingungen, die ein Solver lösen
müsste; die Textsuche ist ein einfacher Durchlauf.